# Training curve analysis

Compares runs by **group**, reading the per-epoch `epoch_metrics.csv` that
`scripts/train_gnn.py` writes to each run's `results/<run_name>/` directory.

Two groups, because they answer different questions and sharing an axis makes both harder to
read:

1. **Aggregation method** — mean pooling vs. MPNN vs. GraphTransformer. Three structurally
   different models on identical windows, splits and LCPN classifier, so the comparison
   isolates aggregation and nothing else.
2. **GraphTransformer switches** — the GT baseline plus its single-switch ablations. Every
   entry differs from one shared baseline by exactly one switch, so the meaningful quantity is
   the **delta from baseline**, not the absolute height of a bar. Section 2 plots both.

There is deliberately no single-model spotlight section: a group comparison already contains
everything a one-model view would show, and the per-model curves in section 3 cover
convergence.

CSV columns: `epoch`, `train_loss`, `window_*` and `cell_*` accuracy / balanced_accuracy /
macro_precision / macro_f1, plus `window_recall_<class>` / `cell_recall_<class>` and the
matching `_precision_` columns per class. Per-class F1 is not logged — it's derived here from
precision + recall (`f1_from_pr`) rather than duplicated in the CSV.

Kernel: **segclr_db (.venv)** — needs pandas + matplotlib
(`scripts/sbatch/install_matplotlib.sh`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path("..")
RESULTS_DIR = REPO_ROOT / "results"

# ---------------------------------------------------------------------------
# Configuration -- edit the two groups below and re-run.
#
# Run names encode the aggregation method (scripts/train_gnn.py's agg_tag):
# "meanpool", "mpnn_L{layers}", or "gt_L{depth}_H{heads}" with any enabled
# switch appended (_nolpe, _norelpos, _noadjbias, _nbhd, _distbias, _thick).
#
# The GT baseline appears in both groups deliberately: it is the third
# architecture AND the reference point for every ablation.
# ---------------------------------------------------------------------------

GT_BASELINE_LABEL = "GT baseline"
_GT_BASELINE_CSV = RESULTS_DIR / "gnn_lcpn_scratch_gt_L4_H4" / "epoch_metrics.csv"

ARCH_CSVS = [
    ("mean pooling", RESULTS_DIR / "gnn_lcpn_scratch_meanpool" / "epoch_metrics.csv"),
    ("MPNN L2", RESULTS_DIR / "gnn_lcpn_scratch_mpnn_L2" / "epoch_metrics.csv"),
    (GT_BASELINE_LABEL, _GT_BASELINE_CSV),
]

# Baseline FIRST -- the delta plots in section 2 take the first entry as the
# reference point.
GT_CSVS = [
    (GT_BASELINE_LABEL, _GT_BASELINE_CSV),
    ("no LPE", RESULTS_DIR / "gnn_lcpn_scratch_gt_L4_H4_nolpe" / "epoch_metrics.csv"),
    ("neighborhood attn", RESULTS_DIR / "gnn_lcpn_scratch_gt_L4_H4_nbhd" / "epoch_metrics.csv"),
    ("no adj bias", RESULTS_DIR / "gnn_lcpn_scratch_gt_L4_H4_noadjbias" / "epoch_metrics.csv"),
    ("no rel_pos", RESULTS_DIR / "gnn_lcpn_scratch_gt_L4_H4_norelpos" / "epoch_metrics.csv"),
    ("+ dist bias", RESULTS_DIR / "gnn_lcpn_scratch_gt_L4_H4_distbias" / "epoch_metrics.csv"),
]

COMPARISON_GROUPS = [
    ("Aggregation method", ARCH_CSVS),
    ("GraphTransformer switches", GT_CSVS),
]

# Which metric selects each run's "best" epoch -- matches train_gnn.py's own
# checkpoint-selection criterion (best val_window_bacc: more stable than
# cell-level, which majority-votes only a few hundred val cells), so "best
# epoch" here means the same thing as the saved checkpoint_best.pt.
BEST_EPOCH_METRIC = "window_balanced_accuracy"

MANIFEST_PATH = REPO_ROOT / "data" / "manifest.json"  # for the class-support section

plt.rcParams["figure.dpi"] = 110

In [ ]:
def load_metrics(csv_path) -> pd.DataFrame:
    return pd.read_csv(csv_path)


def class_names_from_columns(df: pd.DataFrame, prefix: str = "window_recall_") -> list[str]:
    return [c[len(prefix):] for c in df.columns if c.startswith(prefix)]


def best_epoch_row(df: pd.DataFrame, metric: str = BEST_EPOCH_METRIC) -> pd.Series:
    return df.loc[df[metric].idxmax()]


def f1_from_pr(precision, recall) -> np.ndarray:
    """Per-class F1 from precision + recall -- not logged to the CSV (only the
    macro F1 scalar is), so every plot that wants per-class F1 derives it here
    instead. Matches gnn/metrics.py::macro_f1's harmonic-mean formula, just
    element-wise over classes instead of pre-averaged."""
    p, r = np.asarray(precision, dtype=float), np.asarray(recall, dtype=float)
    denom = p + r
    return np.divide(2 * p * r, denom, out=np.zeros_like(denom), where=denom > 0)


def has_precision_cols(df: pd.DataFrame) -> bool:
    return "window_macro_precision" in df.columns


def available_runs(pairs, group_name=""):
    """Drop entries whose CSV isn't on disk yet, with a note.

    Runs are submitted as a batch and land one at a time, so a group is
    routinely half-complete. Skipping with a printed note beats a
    FileNotFoundError that takes the whole notebook down mid-sweep.
    """
    have = [(label, path) for label, path in pairs if Path(path).exists()]
    missing = [label for label, path in pairs if not Path(path).exists()]
    if missing:
        print(f"  [{group_name}] not on disk yet, skipped: {', '.join(missing)}")
    return have


def grouped_bar(ax, x_labels, series_by_label, colors=None, ylabel="score", title="", ylim=(0, 1)):
    """series_by_label: {series_label: [value per x_label]}. One group of
    adjacent, differently-colored bars per x position."""
    n_series = len(series_by_label)
    x = np.arange(len(x_labels))
    width = 0.8 / max(n_series, 1)
    if colors is None:
        colors = plt.cm.tab10(np.linspace(0, 1, max(n_series, 2)))
    for i, (label, values) in enumerate(series_by_label.items()):
        offset = i * width - (n_series - 1) * width / 2
        ax.bar(x + offset, values, width, label=label, color=colors[i])
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=60 if len(x_labels) > 4 else 0,
                       ha="right" if len(x_labels) > 4 else "center")
    ax.set_ylabel(ylabel)
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.legend(fontsize=8)


def load_group(pairs, group_name="", metric=BEST_EPOCH_METRIC) -> pd.DataFrame:
    """One row per run, taken at that run's own best epoch."""
    rows = []
    for label, path in available_runs(pairs, group_name):
        b = best_epoch_row(load_metrics(path), metric).copy()
        b["model"] = label
        rows.append(b)
    return pd.DataFrame(rows).reset_index(drop=True)

## 1. Headline comparison, per group

Each run evaluated at its **own** best epoch (by `BEST_EPOCH_METRIC`). Window-level metrics
average over ~2.4M test windows; cell-level metrics majority-vote those up to ~439 cells and
are correspondingly noisier.

In [ ]:
group_frames = {}
for group_name, pairs in COMPARISON_GROUPS:
    df_g = load_group(pairs, group_name)
    group_frames[group_name] = df_g
    if df_g.empty:
        print(f"### {group_name}: no runs on disk yet\n")
        continue
    cols = ["model", "epoch", "window_accuracy", "window_balanced_accuracy"]
    cols += [c for c in ("window_macro_precision", "window_macro_f1") if c in df_g.columns]
    cols += ["cell_accuracy", "cell_balanced_accuracy"]
    cols += [c for c in ("cell_macro_precision", "cell_macro_f1") if c in df_g.columns]
    print(f"### {group_name}")
    display(df_g[cols].round(4))

In [ ]:
for group_name, df_g in group_frames.items():
    if df_g.empty:
        continue
    metric_cols = ["window_accuracy", "window_balanced_accuracy"]
    metric_labels = ["window acc", "window balanced acc"]
    if "window_macro_precision" in df_g.columns and df_g["window_macro_precision"].notna().all():
        metric_cols += ["window_macro_precision", "window_macro_f1"]
        metric_labels += ["window precision", "window F1"]
    else:
        metric_cols += ["window_macro_f1"]
        metric_labels += ["window F1"]

    fig, ax = plt.subplots(figsize=(10, 5))
    series = {row["model"]: [row[c] for c in metric_cols] for _, row in df_g.iterrows()}
    grouped_bar(ax, metric_labels, series, title=f"{group_name} (each at its own best epoch)")
    plt.tight_layout()
    plt.show()

## 2. Per-class comparison, per group

Per-class window-level recall, precision and F1 at each run's best epoch.

For the **GraphTransformer switches** group a second view follows: the signed **delta from the
baseline**. Since every entry there differs from the baseline by exactly one switch, the delta
is what the switch actually did — absolute bars mostly show the shared baseline behaviour and
bury the effect being measured.

In [ ]:
def per_class_series(pairs, group_name):
    """{label: [value per class]} for recall / precision / F1, plus the class list."""
    have = available_runs(pairs, group_name)
    if not have:
        return None, None, None, None
    classes = class_names_from_columns(load_metrics(have[0][1]))
    recall, precision, f1 = {}, {}, {}
    for label, path in have:
        d = load_metrics(path)
        b = best_epoch_row(d)
        recall[label] = [b[f"window_recall_{c}"] for c in classes]
        if has_precision_cols(d):
            precision[label] = [b[f"window_precision_{c}"] for c in classes]
            f1[label] = f1_from_pr(precision[label], recall[label])
    return classes, recall, precision, f1


for group_name, pairs in COMPARISON_GROUPS:
    classes, recall, precision, f1 = per_class_series(pairs, group_name)
    if not classes:
        continue
    w = max(10, len(classes) * 0.6)
    for series, ylabel in ((recall, "recall"), (precision, "precision"), (f1, "F1")):
        if not series or len(series) != len(recall):
            print(f"note: [{group_name}] a run predates per-class {ylabel} logging -- skipped.")
            continue
        fig, ax = plt.subplots(figsize=(w, 5))
        grouped_bar(ax, classes, series, ylabel=f"window-level {ylabel}",
                    title=f"{group_name}: per-class window {ylabel}")
        plt.tight_layout()
        plt.show()

In [ ]:
# Delta view -- GraphTransformer switches only, where a shared baseline exists.
classes, recall, precision, f1 = per_class_series(GT_CSVS, "GraphTransformer switches")
if classes and GT_BASELINE_LABEL in recall and len(recall) > 1:
    for series, ylabel in ((recall, "recall"), (f1, "F1")):
        if not series or GT_BASELINE_LABEL not in series:
            continue
        base = np.asarray(series[GT_BASELINE_LABEL], dtype=float)
        deltas = {
            label: np.asarray(vals, dtype=float) - base
            for label, vals in series.items()
            if label != GT_BASELINE_LABEL
        }
        lim = float(np.abs(np.concatenate(list(deltas.values()))).max()) * 1.15 or 0.01
        fig, ax = plt.subplots(figsize=(max(10, len(classes) * 0.6), 5))
        grouped_bar(ax, classes, deltas, ylabel=f"delta window {ylabel} vs baseline",
                    title=f"GraphTransformer switches: change in per-class {ylabel} "
                          f"vs {GT_BASELINE_LABEL}",
                    ylim=(-lim, lim))
        ax.axhline(0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()
else:
    print("delta view needs the GT baseline plus at least one ablation on disk.")

## 3. Training curves, per group

Train loss and validation window balanced accuracy against epoch, one line per run. This is
where convergence and overfitting show up: a run whose train loss keeps falling while its
validation metric flattens has stopped generalizing, and a run still climbing at the last
epoch was cut short rather than converged.

Note "val" is an alias for the test split (see `data/build_dataset_from_store.py`), so these
curves are not held out from the reported numbers — an accepted trade-off of the two-way
split.

In [ ]:
for group_name, pairs in COMPARISON_GROUPS:
    have = available_runs(pairs, group_name)
    if not have:
        continue
    fig, (ax_loss, ax_bacc) = plt.subplots(1, 2, figsize=(13, 4.5))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(have), 2)))
    for (label, path), color in zip(have, colors):
        d = load_metrics(path)
        ax_loss.plot(d["epoch"], d["train_loss"], label=label, color=color)
        ax_bacc.plot(d["epoch"], d["window_balanced_accuracy"], label=label, color=color)
    ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("train loss")
    ax_loss.set_title(f"{group_name}: train loss")
    ax_bacc.set_xlabel("epoch"); ax_bacc.set_ylabel("val window balanced accuracy")
    ax_bacc.set_title(f"{group_name}: val window balanced accuracy")
    for a in (ax_loss, ax_bacc):
        a.legend(fontsize=8)
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Class support (train-split window counts)

Context for the per-class bars above — a class with very few training windows can swing wildly
on recall from run to run purely from noise, which the recall number alone doesn't show. Reads
`data/manifest.json` directly (same computation as
`data/dataset_lcpn.py::train_window_counts_by_label`).

In [ ]:
def train_window_counts_by_label(manifest_path) -> dict[str, int]:
    manifest = json.loads(Path(manifest_path).read_text())
    counts: dict[str, int] = {}
    for info in manifest["cells"].values():
        if info["split"] == "train":
            counts[info["cell_type"]] = counts.get(info["cell_type"], 0) + info["n_nodes_covered"]
    return counts


_first = next((p for _, p in ARCH_CSVS + GT_CSVS if Path(p).exists()), None)
if _first is None:
    print("no runs on disk yet -- class list comes from a run's CSV columns.")
else:
    classes = class_names_from_columns(load_metrics(_first))
    support = train_window_counts_by_label(MANIFEST_PATH)
    fig, ax = plt.subplots(figsize=(max(8, len(classes) * 0.5), 4))
    ax.bar(classes, [support.get(c, 0) for c in classes], color="#55A868")
    ax.set_yscale("log")
    ax.set_ylabel("train window count (log scale)")
    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=60, ha="right")
    ax.set_title("Class support (train split)")
    plt.tight_layout()
    plt.show()